# Viral Distance Calculation

First, we need to align both samples to enable a basepair-wise comparison.

In [1134]:
# importing the module
import json

with open('data/sample1.json') as json_file:
    sample1_mutations = json.load(json_file)

with open('data/sample2.json') as json_file:
    sample2_mutations = json.load(json_file)

In [1135]:
sample1_mutations, sample2_mutations

({'substitutions': [{'pos': 209, 'refNuc': 'G', 'qryNuc': 'T'},
   {'pos': 240, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 244, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 245, 'refNuc': 'G', 'qryNuc': 'T'},
   {'pos': 3036, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 4180, 'refNuc': 'G', 'qryNuc': 'T'},
   {'pos': 4183, 'refNuc': 'G', 'qryNuc': 'A'},
   {'pos': 6401, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 6658, 'refNuc': 'A', 'qryNuc': 'G'},
   {'pos': 7123, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 8985, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 9052, 'refNuc': 'G', 'qryNuc': 'T'},
   {'pos': 10028, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 11200, 'refNuc': 'A', 'qryNuc': 'G'},
   {'pos': 11331, 'refNuc': 'A', 'qryNuc': 'G'},
   {'pos': 14407, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 15450, 'refNuc': 'G', 'qryNuc': 'A'},
   {'pos': 16465, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 19219, 'refNuc': 'C', 'qryNuc': 'T'},
   {'pos': 21617, 'refNuc': 'C', 'qryNuc': 'G'},
   {'pos': 21646, '

In [1136]:
with open('data/reference_string.txt') as reference_string_file:
    reference_string = reference_string_file.read()

### Collect Positions
Group all mutations by their positions in a dictionary. 

In [1137]:
def add_to_dict(dict, pos, type, char):
    if pos not in dict:
        dict[pos] = {type: char}
    else:
        dict[pos][type] = char
    return dict


def get_position_dict(mutations):
    positions = {}
    for insertion in mutations['insertions']:
        positions = add_to_dict(positions, insertion["pos"], "ins", insertion["ins"])
    for substitution in mutations['substitutions']:
        positions = add_to_dict(positions, substitution["pos"], "snp", substitution["qryNuc"])
    for missing in mutations['missing']:
        start = missing["range"]["begin"]
        end = missing["range"]["end"]
        character = missing["character"]
        for x in range(start, end):
            positions = add_to_dict(positions, x, "snp", character)
    for missing in mutations['nonACGTNs']:
        start = missing["range"]["begin"]
        end = missing["range"]["end"]
        character = missing["character"]
        for x in range(start, end):
            positions = add_to_dict(positions, x, "snp", character)
    for deletion in mutations['deletions']:
        start = deletion["range"]["begin"]
        end = deletion["range"]["end"]
        for x in range(start, end):
            positions = add_to_dict(positions, x, "del", "-")
    alignmentStart = mutations["alignmentRange"]["begin"]
    alignmentEnd = mutations["alignmentRange"]["end"]
    for x in range(0, alignmentStart):
        positions = add_to_dict(positions, x, "del", "-")

    for x in range(alignmentEnd, len(reference_string)):
        positions = add_to_dict(positions, x, "del", "-")

    return dict(sorted(positions.items()))

In [1138]:
get_position_dict(sample2_mutations)

{0: {'snp': 'N'},
 1: {'snp': 'N'},
 2: {'snp': 'N'},
 3: {'snp': 'N'},
 4: {'snp': 'N'},
 5: {'snp': 'N'},
 6: {'snp': 'N'},
 7: {'snp': 'N'},
 8: {'snp': 'N'},
 9: {'snp': 'N'},
 10: {'snp': 'N'},
 11: {'snp': 'N'},
 12: {'snp': 'N'},
 13: {'snp': 'N'},
 14: {'snp': 'N'},
 15: {'snp': 'N'},
 16: {'snp': 'N'},
 17: {'snp': 'N'},
 18: {'snp': 'N'},
 19: {'snp': 'N'},
 20: {'snp': 'N'},
 21: {'snp': 'N'},
 22: {'snp': 'N'},
 23: {'snp': 'N'},
 24: {'snp': 'N'},
 25: {'snp': 'N'},
 26: {'snp': 'N'},
 27: {'snp': 'N'},
 28: {'snp': 'N'},
 29: {'snp': 'N'},
 30: {'snp': 'N'},
 31: {'snp': 'N'},
 32: {'snp': 'N'},
 33: {'snp': 'N'},
 34: {'snp': 'N'},
 35: {'snp': 'N'},
 36: {'snp': 'N'},
 37: {'snp': 'N'},
 38: {'snp': 'N'},
 39: {'snp': 'N'},
 40: {'snp': 'N'},
 41: {'snp': 'N'},
 42: {'snp': 'N'},
 43: {'snp': 'N'},
 44: {'snp': 'N'},
 45: {'snp': 'N'},
 46: {'snp': 'N'},
 47: {'snp': 'N'},
 48: {'snp': 'N'},
 49: {'snp': 'N'},
 50: {'snp': 'N'},
 51: {'snp': 'N'},
 52: {'snp': 'N'},
 53

### Align Samples

In [1139]:
from Bio import Align

aligner = Align.PairwiseAligner(match_score=1.0)
test1 = "GAACT"
test2 = "GAT"
alignments = aligner.align(test1, test2)
alignments[0][0], alignments[1][1]

('GAACT', 'G-A-T')

In [1140]:
def align_samples(sample1_mutations, sample2_mutations):
    sequence1 = ""
    sequence2 = ""
    positions1 = get_position_dict(sample1_mutations)
    positions2 = get_position_dict(sample2_mutations)
    for current_base_index in range(0, len(reference_string) - 1):
        reference_index_char = reference_string[current_base_index]
        additions1 = ""
        additions2 = ""

        # add remaining reference characters if index is not in positions dictionary
        if current_base_index not in positions1:
            additions1 += reference_index_char + additions1
        if current_base_index not in positions2:
            additions2 += reference_index_char + additions2

        # get mutations for current base index or return empty list if no mutations for current index exist
        position_mutations1 = positions1[current_base_index] if current_base_index in positions1 else {}
        position_mutations2 = positions2[current_base_index] if current_base_index in positions2 else {}

        # add snp characters for both sequences
        additions1 += position_mutations1["snp"] if "snp" in position_mutations1 else ""
        additions2 += position_mutations2["snp"] if "snp" in position_mutations2 else ""

        # handle deletions for current base index, respecting the others sequence mutations
        if not ("del" in position_mutations1 and "del" in position_mutations2) and not (
                "del" not in position_mutations1 and "del" not in position_mutations2):
            additions1 += "-" if "del" in position_mutations1 else ""
            additions2 += "-" if "del" in position_mutations2 else ""

        # handle insertions
        if "ins" in position_mutations1 and "ins" in position_mutations2:
            if position_mutations1["ins"] != position_mutations2["ins"]:
                alignments = aligner.align(position_mutations1["ins"], position_mutations2["ins"])
                additions1 += alignments[0][0]
                additions2 += alignments[0][1]
                print(additions1, additions2)
            else:
                additions1 += position_mutations1["ins"] if len(
                    position_mutations1) > 1 else reference_index_char + additions1 + position_mutations1["ins"]
                additions2 += position_mutations2["ins"] if len(
                    position_mutations2) > 1 else reference_index_char + additions2 + position_mutations2["ins"]
        elif "ins" in position_mutations1:
            additions1 += position_mutations1["ins"] if len(
                position_mutations1) > 1 else reference_index_char + additions1 + position_mutations1["ins"]
            additions2 += "-" * len(position_mutations1["ins"])
        elif "ins" in position_mutations2:
            additions1 += "-" * len(position_mutations2["ins"])
            additions2 += position_mutations2["ins"] if len(
                position_mutations2) > 1 else reference_index_char + additions2 + position_mutations2["ins"]

        # add new additions to prior sequences
        sequence1 += additions1
        sequence2 += additions2
    return sequence1, sequence2

In [1141]:
sequence1, sequence2 = align_samples(sample1_mutations, sample2_mutations)

In [1142]:
sequence1

'...'

### Alignment validation

Check equal length of sequence results

In [1143]:
len(sequence1), len(sequence2)

(29895, 29895)

Validate single sequence insertions without alignment

In [1144]:
sequence1[155:160], sequence2[155:160]

('ACAGG', 'ACAGG')

Validate multi sequence insertions alignment

In [1145]:
sequence1[100:105], sequence2[100:105]

('GGCTG', 'GGCTG')

Validate single sequence deletion

In [1146]:
sequence1[28258:28264], sequence2[28258:28264]

('CAAACT', 'CAAACT')

### Distance calculation

In [1147]:
ambiguous_characters = {
    "A": ["A"],
    "C": ["C"],
    "G": ["G"],
    "T": ["T"],
    "U": ["U"],
    "M": ["A", "C"],
    "R": ["A", "G"],
    "S": ["C", "G"],
    "W": ["A", "T"],
    "Y": ["C", "T"],
    "K": ["G", "T"],
    "V": ["A", "C", "G"],
    "H": ["A", "C", "T"],
    "D": ["A", "G", "T"],
    "B": ["C", "G", "T"],
    "N": ["A", "C", "G", "T"],
    "X": ["A", "C", "G", "T"],
}
 
def calculate_distance(sequence1, sequence2):
    distance = 0
    proper_threshold = 5
    proper_chars_1 = 0
    proper_chars_2 = 0
    n_count_1 = sequence1.count('N')
    n_count_2 = sequence2.count('N')
    sequence_length_1 = len(sequence1)
    sequence_length_2 = len(sequence2)
    active_gap_1 = False
    active_gap_2 = False
    for current_base_index in range(0, sequence_length_1):
        current_char_1 = sequence1[current_base_index]
        current_char_2 = sequence2[current_base_index]
        if current_char_1 != "-":
            active_gap_1 = False
        if current_char_2 != "-":
            active_gap_2 = False

        # dont increment distance, dont increment proper_chars
        if current_char_1 == "N" or current_char_2 == "N":
            continue

        # increment proper_chars if current char is not "-"
        proper_chars_1 += 1 if current_char_1 != "-" else 0
        proper_chars_2 += 1 if current_char_2 != "-" else 0

        # dont increment distance
        if current_char_1 == current_char_2:
            continue
        
        # continue if amount of proper chars is not yet reached to increment distances
        if (proper_chars_1 < proper_threshold) or (
                (sequence_length_1 - n_count_1) - proper_chars_1 < proper_threshold) or (
                proper_chars_2 < proper_threshold) or (
                (sequence_length_2 - n_count_2) - proper_chars_2 < proper_threshold):
            continue

        # increment distance on gap
        if current_char_1 == "-":
            if not active_gap_1:
                distance += 1
        if current_char_2 == "-":
            if not active_gap_2:
                distance += 1
            
        # increment distance if not gap and differing chars
        if current_char_1 != "-" and current_char_2 != "-" and not (current_char_1 in ambiguous_characters[current_char_2] or current_char_2 in ambiguous_characters[current_char_1]):
            distance += 1
            
    return distance


Frage Jonas:
- Proper Chars zurücksetzen oder nur nicht hochzählen, wenn N oder - gefunden?

In [1148]:
distance = calculate_distance(sequence1, sequence2)
distance

2

        for (let i = 0; i < sequence1.length; i++) {
            this.currentChar1 = sequence1[i];
            this.currentChar2 = sequence2[i];

            if (this.atleastOneCharIsN()) {
                continue;
            }

            this.incrementProperCharsSeen();

            if (this.properThresholdNotReached() || this.charsAreEqual()) {
                continue;
            }

            this.incrementDistanceOnGaps();

            this.incrementDistanceForDifferingChars();
        }